# 05-06_population_and_global_city_scraping

Population enrichment, scrape timestamps, a global city scraper, and transformation into relational-style city/facts tables.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime

In [ ]:
# Challenge 2.1 continued — helper function to convert DMS (degrees°minutes′seconds″) to decimal
import re

def dms_to_decimal(dms_str):
    """Convert a DMS string like 52°31′12″N to a decimal float like 52.5200"""
    parts = re.split(r'[°′″]', dms_str.strip())
    degrees = float(parts[0])
    minutes = float(parts[1]) if len(parts) > 1 and parts[1] else 0
    seconds = float(parts[2]) if len(parts) > 2 and parts[2] else 0
    direction = parts[-1].strip()  # N, S, E, W
    decimal = degrees + minutes / 60 + seconds / 3600
    if direction in ('S', 'W'):
        decimal *= -1
    return round(decimal, 6)

print(dms_to_decimal(lat_raw))  # Expected: ~52.52
print(dms_to_decimal(lon_raw))  # Expected: ~13.405

In [ ]:
def dms_to_decimal(dms_str):
    parts = re.split(r'[°′″]', dms_str.strip())
    degrees   = float(parts[0])
    minutes   = float(parts[1]) if len(parts) > 1 and parts[1] else 0
    # Only treat parts[2] as seconds if it's actually a number
    seconds   = float(parts[2]) if len(parts) > 2 and parts[2] and not parts[2].strip().isalpha() else 0
    direction = parts[-1].strip()
    decimal   = degrees + minutes / 60 + seconds / 3600
    if direction in ('S', 'W'):
        decimal *= -1
    return round(decimal, 6)

In [ ]:
cities = {
    'Berlin':  'https://en.wikipedia.org/wiki/Berlin',
    'Hamburg': 'https://en.wikipedia.org/wiki/Hamburg',
    'Munich':  'https://en.wikipedia.org/wiki/Munich'
}

city_data = []

for city_name, url in cities.items():
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
    soup = BeautifulSoup(response.text, 'html.parser')
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        print(f'⚠️  No infobox found for {city_name}, skipping')
        continue

    # Country — loop through rows to find the 'Country' label
    for row in infobox.find_all('tr'):
        header = row.find('th')
        if header and 'Country' in header.get_text():
            country = row.find('td').get_text(strip=True)
            break

    lat_raw = soup.find('span', class_='latitude').get_text()
    lon_raw = soup.find('span', class_='longitude').get_text()

    city_data.append({
        'city':      city_name,
        'country':   country,
        'latitude':  dms_to_decimal(lat_raw),
        'longitude': dms_to_decimal(lon_raw)
    })
    print(f'✅ Scraped {city_name}')

print(city_data)

In [ ]:
# Bonus Challenge 3.1 — Add population with a timestamp
from datetime import datetime

def get_population(soup):
    """
    Extracts population from a Wikipedia infobox.
    Wikipedia stores population in a td following a th containing 'Population'.
    """
    infobox = soup.find('table', class_='infobox')
    if infobox is None:
        print(f'⚠️  No infobox found for {city_name}, skipping')
        return None
    for row in infobox.find_all('tr'):
        header = row.find('th')
        if header and 'Population' in header.get_text():
            # The actual number is in the next row's td
            next_row = row.find_next_sibling('tr')
            if next_row:
                td = next_row.find('td')
                if td:
                    # Strip footnotes and whitespace, keep only digits and commas
                    raw = td.get_text(separator=' ', strip=True)
                    number = re.sub(r'[^\d,]', '', raw.split()[0])
                    return number
    return None

In [ ]:
# Bonus Challenge 3.2 — Function that returns city info + population + timestamp
def scrape_cities_with_population(city_urls: dict) -> pd.DataFrame:
    """
    Scrapes country, lat, lon, population, and scrape timestamp.
    Returns a clean DataFrame.
    """
    rows = []
    for city_name, url in city_urls.items():
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(response.text, 'html.parser')
        infobox = soup.find('table', class_='infobox')
        if infobox is None:
            print(f'⚠️  No infobox found for {city_name}, skipping')
            continue

        for row in infobox.find_all('tr'):
            header = row.find('th')
            if header and 'Country' in header.get_text():
                country = row.find('td').get_text(strip=True)
                break

        lat_raw    = soup.find('span', class_='latitude').get_text()
        lon_raw    = soup.find('span', class_='longitude').get_text()
        population = get_population(soup)

        rows.append({
            'city':       city_name,
            'country':    country,
            'latitude':   dms_to_decimal(lat_raw),
            'longitude':  dms_to_decimal(lon_raw),
            'population': population,
            'scraped_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        print(f'✅ Scraped {city_name} — population: {population}')

    return pd.DataFrame(rows)

df_population = scrape_cities_with_population(cities)
df_population

In [ ]:
# Bonus Challenge 4 — Global city scraper (handles different country infoboxes)

def scrape_cities_global(city_urls: dict) -> pd.DataFrame:
    """
    Scrapes country, lat, lon, population for any city worldwide.
    More robust than the German-only version — doesn't hardcode 'Germany'.
    Returns a clean DataFrame.
    """
    def get_country(soup):
        """Find country from infobox — looks for a row labelled 'Country'."""
        infobox = soup.find('table', class_='infobox')
        if infobox is None:
            print(f'⚠️  No infobox found for {city_name}, skipping')
            return None
        if not infobox:
            return None
        for row in infobox.find_all('tr'):
            header = row.find('th')
            if header and 'Country' in header.get_text():
                td = row.find('td')
                if td:
                    return td.get_text(strip=True)
        return None

    def get_coords(soup):
        """Extract decimal lat/lon — falls back to raw text if DMS conversion fails."""
        try:
            lat_raw = soup.find('span', class_='latitude').get_text()
            lon_raw = soup.find('span', class_='longitude').get_text()
            return dms_to_decimal(lat_raw), dms_to_decimal(lon_raw)
        except Exception:
            return None, None

    def get_pop(soup):
        """Extract population — same logic as get_population() above."""
        infobox = soup.find('table', class_='infobox')
        if infobox is None:
            print(f'⚠️  No infobox found for {city_name}, skipping')
            return None
        if not infobox:
            return None
        for row in infobox.find_all('tr'):
            header = row.find('th')
            if header and 'Population' in header.get_text():
                next_row = row.find_next_sibling('tr')
                if next_row:
                    td = next_row.find('td')
                    if td:
                        raw = td.get_text(separator=' ', strip=True)
                        number = re.sub(r'[^\d,]', '', raw.split()[0])
                        return number
        return None

    rows = []
    for city_name, url in city_urls.items():
        try:
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
            soup     = BeautifulSoup(response.text, 'html.parser')
            lat, lon = get_coords(soup)
            rows.append({
                'city':       city_name,
                'country':    get_country(soup),
                'latitude':   lat,
                'longitude':  lon,
                'population': get_pop(soup),
                'scraped_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            print(f'✅ Scraped {city_name}')
        except Exception as e:
            print(f'❌ Failed {city_name}: {e}')
    return pd.DataFrame(rows)


# Test with a global mix of cities
global_cities = {
    'Berlin':    'https://en.wikipedia.org/wiki/Berlin',
    'Tokyo':     'https://en.wikipedia.org/wiki/Tokyo',
    'New York':  'https://en.wikipedia.org/wiki/New_York_City',
    'Lagos':     'https://en.wikipedia.org/wiki/Lagos',
    'São Paulo': 'https://en.wikipedia.org/wiki/S%C3%A3o_Paulo'
}

df_global = scrape_cities_global(global_cities)
df_global

In [ ]:
import pandas as pd

# Recreate or reuse df_global from your notebook
# cities table — just city_id and city name
cities_df = df_global[['city']].copy()
cities_df.insert(0, 'city_id', range(1, len(cities_df) + 1))

# facts table — city_id + the facts
facts_df = df_global[['city']].copy()
facts_df.insert(0, 'city_id', range(1, len(facts_df) + 1))
facts_df = facts_df.drop(columns=['city'])
facts_df[['country', 'latitude', 'longitude', 'population', 'scraped_at']] = df_global[['country', 'latitude', 'longitude', 'population', 'scraped_at']]

print(cities_df)
print(facts_df)